**模型**（model）是智能体的大脑，负责推理分析。

**工具**（tools）是智能体的手脚，负责执行任务，与外界交互

定义一个带有工具的Agent的基本流程如下：

定义工具

初始化模型

初始化Agent，绑定模型和工具

1.自定义工具

所谓的工具（**tools**），本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。

包括以下信息：

工具名

工具的作用

工具需要的参数

1.1.基于tool描述工具

可以通过**装饰器**来定义工具名、工具的作用

In [1]:
from langchain_core.tools import tool

@tool("square_root", description = "Calculate the square root of a number")
def tool(x: float) -> float:
    return x ** 0.5

1.2.使用**函数名**和**文档注释**描述工具

如果不@tool装饰器没有定义工具名和描述作用，此时：

工具名：默认就是函数名

工具所需的参数： 默认就是函数的参数列表

工具作用的描述： 默认就是函数的文档注释

In [8]:
from langchain_core.tools import tool

@tool
def square_root(x: float) -> float:
    """
    Calculate the square root of a number
    """
    return x ** 0.5

In [9]:
response = square_root.invoke({"x": 255})
print(response)

15.968719422671311


In [4]:
# 定义一个查询天气的tool

@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    get current weather and optional forecast.
    args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

1.3定义Pydantic Model 描述参数

In [6]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal

# 查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location: str = Field(description = "City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default = "celsius",
        description = "Temperature unit preference"
    )
    include_forecast: bool = Field(
        default = False,
        description = "Include 5-day forecast"
    )


@tool(args_schema = WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    get current weather and optional forecast.
    """
    temp = 22 if units == "celsius" else 72
    result = f"current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [11]:
response = get_weather.invoke({"location": "杭州", "include_forecast": True})
print(response)

current weather in 杭州: 22 degrees C
Next 5 days: Sunny


测试

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = "deepseek-v4-pro",
    tools = [square_root, get_weather]
)

In [14]:
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "杭州接下来的几天天气如何？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

好的，我来帮你查询杭州未来几天的天气。current weather in 杭州: 22 degrees C
Next 5 days: Sunny杭州的天气情况如下：

- **当前气温**：22°C
- **未来5天天气**：晴天 ☀️

总体来看，杭州接下来几天天气不错，以晴好为主，非常适合出行和户外活动。不过早晚温差可能会稍大，建议出门时备一件薄外套。

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content = "467和255的平方根是多少 ？")]}
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和255的平方根是多少 ？
None
================================== Ai Message ==================================

我来计算467和255的平方根。
Tool Calls:
  square_root (call_00_qVfVW7QMln12gX7JCv8n1931)
 Call ID: call_00_qVfVW7QMln12gX7JCv8n1931
  Args:
    x: 467
  square_root (call_01_pAEvdX0LOcZJY7giH7cc9404)
 Call ID: call_01_pAEvdX0LOcZJY7giH7cc9404
  Args:
    x: 255
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

15.968719422671311
None
================================== Ai Message ==================================

计算结果如下：

- **467 的平方根** ≈ **21.6102**
- **255 的平方根** ≈ **15.9687**

如果需要更高精度或进行其他运算，请告诉我！
None
